# SQLite with Python — From Tiny Toy Box to Real Database

> **Goal:** learn how Python talks to SQLite, from the first connection to joins, transactions, and safe error handling.

## Explain it like I am 5

A **database** is a very organized toy cupboard. A **table** is one labeled drawer. Each **row** is one toy card, and each **column** is a fact on that card. SQL is the set of polite instructions we give the cupboard: “add this,” “find that,” or “change this one.”

This notebook keeps the original employee and sales examples, expands them safely, and reads the existing `example.db` and `sales_data.db` files without changing them.

## Learning map

| Level | You will learn |
|---|---|
| Basic | connect, create tables, insert, select, update, delete |
| Intermediate | filtering, sorting, grouping, constraints, joins, row factories |
| Advanced | parameterized queries, transactions, rollback, errors, indexes, backups |

### SQL vocabulary

| Word | Tiny explanation |
|---|---|
| Database | The whole organized cupboard |
| Table | One drawer containing similar records |
| Row | One complete record |
| Column | One kind of fact, such as `name` |
| Primary key | A unique name tag for each row |
| Query | A question or command written in SQL |
| Cursor | Python's helper that carries SQL to SQLite |
| Transaction | A group of changes that succeeds or fails together |

## 1. Setup and locate the existing files

`sqlite3` is part of Python's standard library, so nothing needs to be installed. `Path` gives us reliable file paths.

In [1]:
import sqlite3
from pathlib import Path
from shutil import copy2
from tempfile import TemporaryDirectory

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "example.db").exists():
    candidate = NOTEBOOK_DIR / "Complete-Python-Bootcamp-main" / "11-Working With Databases"
    if candidate.exists():
        NOTEBOOK_DIR = candidate

example_db_path = NOTEBOOK_DIR / "example.db"
sales_db_path = NOTEBOOK_DIR / "sales_data.db"

print("SQLite version:", sqlite3.sqlite_version)
print("Existing files:", example_db_path.name, "and", sales_db_path.name)

SQLite version: 3.53.2
Existing files: example.db and sales_data.db


## 2. Connect to SQLite

The original lesson used `sqlite3.connect('example.db')`. That is correct, but it opens the real file for writing. For a tutorial, we first inspect it in **read-only mode**, so an accidental command cannot change it.

- `connection` represents the open database.
- `cursor` sends commands and receives results.
- `mode=ro` means read only.

In [2]:
def open_read_only(path):
    return sqlite3.connect(f"file:{path.resolve().as_posix()}?mode=ro", uri=True)

connection = open_read_only(example_db_path)
try:
    cursor = connection.cursor()
    tables = cursor.execute(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
    ).fetchall()
    print("Tables in example.db:", tables)
finally:
    connection.close()

Tables in example.db: [('employees',)]


### Context managers close connections safely

`with sqlite3.connect(...) as connection:` handles commit or rollback, but the connection should still be explicitly closed when its lifetime ends. `sqlite3.closing` is not provided, so for this notebook we either use short `with` blocks followed by `close()`, or an in-memory connection closed at the end.

> **Common mistake from the original notebook:** querying with a cursor *after* `connection.close()`. A cursor belongs to its connection; once the connection closes, that cursor cannot work.

## 3. Build a clean practice database

We use `:memory:`. This database lives only in RAM and disappears when closed—perfect for practice. The original `employees` table is preserved and improved with constraints.

### Common column constraints

| Constraint | Meaning |
|---|---|
| `PRIMARY KEY` | unique row identity; SQLite auto-generates integer values |
| `NOT NULL` | value is required |
| `UNIQUE` | duplicates are forbidden |
| `CHECK` | value must pass a rule |
| `DEFAULT` | value used when none is supplied |
| `FOREIGN KEY` | value must point to a row in another table |

In [3]:
connection = sqlite3.connect(":memory:")
connection.execute("PRAGMA foreign_keys = ON")  # SQLite requires this per connection.
cursor = connection.cursor()

cursor.executescript('''
CREATE TABLE departments (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE
);

CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    age INTEGER CHECK (age >= 16),
    department_id INTEGER,
    salary REAL NOT NULL DEFAULT 0 CHECK (salary >= 0),
    active INTEGER NOT NULL DEFAULT 1 CHECK (active IN (0, 1)),
    FOREIGN KEY (department_id) REFERENCES departments(id)
);
''')
connection.commit()
print("Created departments and employees tables.")

Created departments and employees tables.


### Step-by-step

1. `CREATE TABLE` makes a new drawer.
2. `IF NOT EXISTS` can prevent an error when a table already exists; here a clean in-memory database makes it unnecessary.
3. SQLite uses flexible storage classes: `NULL`, `INTEGER`, `REAL`, `TEXT`, and `BLOB`.
4. Declared types are **type affinities**, not always strict rules. Use constraints—or `STRICT` tables in modern SQLite—when stronger checking matters.

## 4. INSERT data safely

The original examples inserted Krish, Bob, and Charlie. We keep them, but use `?` placeholders. A placeholder is a sealed lunchbox: SQLite receives the value separately from the SQL instruction, preventing SQL injection and quoting mistakes.

In [4]:
departments = [(1, "Data Science"), (2, "Engineering"), (3, "Finance")]
cursor.executemany(
    "INSERT INTO departments (id, name) VALUES (?, ?)",
    departments,
)

employees = [
    ("Krish", 32, 1, 90000),
    ("Bob", 25, 2, 72000),
    ("Charlie", 35, 3, 81000),
    ("Asha", 29, 1, 88000),
    ("Meera", 41, 2, 105000),
]
cursor.executemany(
    "INSERT INTO employees (name, age, department_id, salary) VALUES (?, ?, ?, ?)",
    employees,
)
connection.commit()

print("Inserted rows:", cursor.rowcount)
print("Last generated id:", cursor.lastrowid)  # Most useful after a single execute().

Inserted rows: 5
Last generated id: None


### `execute()` vs `executemany()` vs `executescript()`

| Method | Best for |
|---|---|
| `execute(sql, values)` | one SQL statement |
| `executemany(sql, rows)` | same statement for many rows |
| `executescript(sql_text)` | multiple trusted SQL statements; never use with untrusted input |

Never build SQL with f-strings for values. `f"... WHERE name='{user_input}'"` can be unsafe. Placeholders work for **values**, not table or column names; allow-list identifiers separately.

## 5. SELECT and fetch data

`SELECT` asks for data. A cursor does not normally hand you every row immediately; you choose how to fetch it.

In [5]:
cursor.execute("SELECT id, name, age, salary FROM employees ORDER BY id")

first = cursor.fetchone()
next_two = cursor.fetchmany(2)
rest = cursor.fetchall()

print("fetchone():", first)
print("fetchmany(2):", next_two)
print("fetchall():", rest)

fetchone(): (1, 'Krish', 32, 90000.0)
fetchmany(2): [(2, 'Bob', 25, 72000.0), (3, 'Charlie', 35, 81000.0)]
fetchall(): [(4, 'Asha', 29, 88000.0), (5, 'Meera', 41, 105000.0)]


### Fetching choices

| Method | Returns | Use when |
|---|---|---|
| `fetchone()` | one row or `None` | expecting one result |
| `fetchmany(n)` | up to `n` rows | processing chunks |
| `fetchall()` | all remaining rows | result is known to be small |
| iterate cursor | rows one at a time | result might be large |

Calling one fetch method moves the cursor forward. That is why each call above returned the *next* rows.

## 6. WHERE, ORDER BY, and LIMIT

- `WHERE` keeps matching rows.
- `ORDER BY` sorts them (`ASC` up, `DESC` down).
- `LIMIT` stops after a chosen number.

SQL's logical idea is “filter, sort, take a few,” even though we write `SELECT ... WHERE ... ORDER BY ... LIMIT ...`.

In [6]:
minimum_salary = 80000
rows = cursor.execute(
    '''
    SELECT name, age, salary
    FROM employees
    WHERE salary >= ? AND active = 1
    ORDER BY salary DESC, name ASC
    LIMIT 3
    ''',
    (minimum_salary,),  # A one-item tuple needs the comma.
).fetchall()

for row in rows:
    print(row)

('Meera', 41, 105000.0)
('Krish', 32, 90000.0)
('Asha', 29, 88000.0)


### Useful filters and NULL's special rule

| Need | SQL example |
|---|---|
| one of several values | `WHERE department_id IN (1, 3)` |
| a range | `WHERE age BETWEEN 25 AND 35` |
| a text pattern | `WHERE name LIKE 'A%'` |
| missing value | `WHERE department_id IS NULL` |
| combine rules | `WHERE active = 1 AND salary > 80000` |

`NULL` means “unknown/missing.” Do **not** write `= NULL`; use `IS NULL` or `IS NOT NULL`.

## 7. UPDATE and DELETE — preserved original examples

The original notebook changed Krish's age to 34 and deleted Bob. We perform the same actions on our disposable practice database.

> Before `UPDATE` or `DELETE`, run the same `WHERE` with `SELECT`. A missing `WHERE` changes every row.

In [7]:
# Preview the row first.
print(cursor.execute(
    "SELECT id, name, age FROM employees WHERE name = ?", ("Krish",)
).fetchall())

cursor.execute(
    "UPDATE employees SET age = ? WHERE name = ?",
    (34, "Krish"),
)
print("Rows updated:", cursor.rowcount)

cursor.execute("DELETE FROM employees WHERE name = ?", ("Bob",))
print("Rows deleted:", cursor.rowcount)
connection.commit()

print(cursor.execute("SELECT id, name, age FROM employees ORDER BY id").fetchall())

[(1, 'Krish', 32)]
Rows updated: 1
Rows deleted: 1
[(1, 'Krish', 34), (3, 'Charlie', 35), (4, 'Asha', 29), (5, 'Meera', 41)]


## 8. Aggregate functions and GROUP BY

An aggregate squishes many rows into a summary number.

| Function | Job |
|---|---|
| `COUNT(*)` | count rows |
| `SUM(column)` | add values |
| `AVG(column)` | average values |
| `MIN(column)` | smallest value |
| `MAX(column)` | largest value |

`GROUP BY` makes one summary per group. `HAVING` filters groups after aggregation; `WHERE` filters rows before aggregation.

In [8]:
summary = cursor.execute('''
    SELECT d.name AS department,
           COUNT(e.id) AS employee_count,
           ROUND(AVG(e.salary), 2) AS average_salary,
           MAX(e.salary) AS highest_salary
    FROM departments AS d
    LEFT JOIN employees AS e ON e.department_id = d.id
    GROUP BY d.id, d.name
    HAVING COUNT(e.id) >= 1
    ORDER BY average_salary DESC
''').fetchall()

for row in summary:
    print(row)

('Engineering', 1, 105000.0, 105000.0)
('Data Science', 2, 89000.0, 90000.0)
('Finance', 1, 81000.0, 81000.0)


## 9. JOIN tables

A join matches cards from two drawers.

| Join | Keeps |
|---|---|
| `INNER JOIN` | only matching rows from both sides |
| `LEFT JOIN` | every left row, plus matches from the right |
| `CROSS JOIN` | every possible pair—can become huge |

SQLite supports `INNER`, `LEFT`, `CROSS`, and modern versions also support `RIGHT`/`FULL`; `LEFT JOIN` remains the most portable outer join choice.

In [9]:
joined_rows = cursor.execute('''
    SELECT e.name AS employee, d.name AS department, e.salary
    FROM employees AS e
    INNER JOIN departments AS d ON d.id = e.department_id
    ORDER BY department, employee
''').fetchall()

for row in joined_rows:
    print(row)

('Asha', 'Data Science', 88000.0)
('Krish', 'Data Science', 90000.0)
('Meera', 'Engineering', 105000.0)
('Charlie', 'Finance', 81000.0)


## 10. Friendlier rows with `sqlite3.Row`

Tuple positions like `row[2]` are easy to forget. A row factory lets us use column names like labels on boxes.

In [10]:
connection.row_factory = sqlite3.Row
named_row = connection.execute(
    "SELECT id, name, salary FROM employees WHERE name = ?", ("Asha",)
).fetchone()

print(dict(named_row))
print("Name:", named_row["name"], "| Salary:", named_row["salary"])

{'id': 4, 'name': 'Asha', 'salary': 88000.0}
Name: Asha | Salary: 88000.0


## 11. Use the existing `sales_data.db` safely

The original notebook created a `sales` table and inserted five rows. The repository already contains that database, so we read its real schema and data in read-only mode. Results are bounded with `LIMIT`.

In [11]:
sales_connection = open_read_only(sales_db_path)
try:
    sales_connection.row_factory = sqlite3.Row
    schema = sales_connection.execute(
        "SELECT sql FROM sqlite_master WHERE type='table' AND name='sales'"
    ).fetchone()
    print("Existing schema:\n", schema[0])

    sample = sales_connection.execute(
        "SELECT id, date, product, sales, region FROM sales ORDER BY id LIMIT 5"
    ).fetchall()
    print("\nSample rows:")
    for row in sample:
        print(dict(row))
finally:
    sales_connection.close()

Existing schema:
 CREATE TABLE sales (
    id INTEGER PRIMARY KEY,
    date TEXT NOT NULL,
    product TEXT NOT NULL,
    sales INTEGER,
    region TEXT
)

Sample rows:
{'id': 1, 'date': '2023-01-01', 'product': 'Product1', 'sales': 100, 'region': 'North'}
{'id': 2, 'date': '2023-01-02', 'product': 'Product2', 'sales': 200, 'region': 'South'}
{'id': 3, 'date': '2023-01-03', 'product': 'Product1', 'sales': 150, 'region': 'East'}
{'id': 4, 'date': '2023-01-04', 'product': 'Product3', 'sales': 250, 'region': 'West'}
{'id': 5, 'date': '2023-01-05', 'product': 'Product2', 'sales': 300, 'region': 'North'}


### Practical sales report

We summarize existing data without assuming how many times the old notebook may have been run. `COUNT`, `SUM`, and `AVG` turn detailed sales rows into a report.

In [12]:
sales_connection = open_read_only(sales_db_path)
try:
    report = sales_connection.execute('''
        SELECT region,
               COUNT(*) AS sale_count,
               SUM(sales) AS total_sales,
               ROUND(AVG(sales), 2) AS average_sale
        FROM sales
        GROUP BY region
        ORDER BY total_sales DESC, region
    ''').fetchall()
finally:
    sales_connection.close()

for row in report:
    print(row)

('North', 2, 400, 200.0)
('West', 1, 250, 250.0)
('South', 1, 200, 200.0)
('East', 1, 150, 150.0)


## 12. Transactions: all together or not at all

Imagine moving 100 coins from Asha's jar to Meera's jar. We must not remove coins from one jar unless we can add them to the other. A transaction groups both changes.

- `commit()` makes changes permanent.
- `rollback()` returns to the state before the transaction.
- Python's connection context manager commits on success and rolls back when an exception escapes the block.

In [13]:
connection.row_factory = None
before = connection.execute(
    "SELECT name, salary FROM employees WHERE name IN (?, ?) ORDER BY name",
    ("Asha", "Meera"),
).fetchall()

try:
    connection.execute("BEGIN")
    connection.execute("UPDATE employees SET salary = salary - ? WHERE name = ?", (1000, "Asha"))
    connection.execute("UPDATE employees SET salary = salary + ? WHERE name = ?", (1000, "Meera"))
    connection.commit()
except sqlite3.Error:
    connection.rollback()
    raise

after = connection.execute(
    "SELECT name, salary FROM employees WHERE name IN (?, ?) ORDER BY name",
    ("Asha", "Meera"),
).fetchall()
print("Before:", before)
print("After: ", after)

Before: [('Asha', 88000.0), ('Meera', 105000.0)]
After:  [('Asha', 87000.0), ('Meera', 106000.0)]


### See rollback in action

We intentionally violate the `salary >= 0` rule. The exception is caught, and rollback erases the unfinished change.

In [14]:
salary_before = connection.execute(
    "SELECT salary FROM employees WHERE name = ?", ("Krish",)
).fetchone()[0]

try:
    connection.execute("BEGIN")
    connection.execute(
        "UPDATE employees SET salary = ? WHERE name = ?",
        (-1, "Krish"),
    )
    connection.commit()
except sqlite3.IntegrityError as error:
    connection.rollback()
    print("Rolled back safely:", error)

salary_after = connection.execute(
    "SELECT salary FROM employees WHERE name = ?", ("Krish",)
).fetchone()[0]
print("Salary unchanged:", salary_before == salary_after)

Rolled back safely: CHECK constraint failed: salary >= 0
Salary unchanged: True


## 13. SQLite error handling

Catch the narrowest useful exception and keep transaction boundaries clear.

| Exception | Typical cause |
|---|---|
| `sqlite3.IntegrityError` | constraint failed |
| `sqlite3.OperationalError` | bad SQL, missing table, locked database |
| `sqlite3.ProgrammingError` | closed connection/cursor misuse, wrong bindings |
| `sqlite3.DatabaseError` | broader database problem |
| `sqlite3.Error` | base class for SQLite errors |

Do not hide errors with bare `except:`. Log or report enough context, but never expose passwords or sensitive values.

In [15]:
try:
    connection.execute(
        "INSERT INTO departments (name) VALUES (?)",
        ("Engineering",),  # UNIQUE constraint already has this value.
    )
    connection.commit()
except sqlite3.IntegrityError as error:
    connection.rollback()
    print("Friendly message: that department already exists.")
    print("SQLite detail:", error)

Friendly message: that department already exists.
SQLite detail: UNIQUE constraint failed: departments.name


## 14. Indexes, query plans, and performance

An index is like the alphabet tabs in a dictionary: faster searching, but it uses space and makes writes a little slower. Index columns used often in filters, joins, or sorting—not every column.

In [16]:
connection.execute("CREATE INDEX idx_employees_department ON employees(department_id)")
plan = connection.execute(
    "EXPLAIN QUERY PLAN SELECT * FROM employees WHERE department_id = ?",
    (1,),
).fetchall()
print(plan)

[(3, 0, 61, 'SEARCH employees USING INDEX idx_employees_department (department_id=?)')]


## 15. Practical pattern: a small repository function

Keep SQL inside focused functions. Validate dynamic choices with an allow-list, and keep data values parameterized.

In [17]:
def find_employees(db_connection, minimum_salary=0, sort_by="name"):
    allowed_sort_columns = {"name", "age", "salary"}
    if sort_by not in allowed_sort_columns:
        raise ValueError(f"sort_by must be one of {sorted(allowed_sort_columns)}")

    # The column is safe because it came from our allow-list; the value uses ?.
    sql = f'''
        SELECT name, age, salary
        FROM employees
        WHERE active = 1 AND salary >= ?
        ORDER BY {sort_by}
    '''
    return db_connection.execute(sql, (minimum_salary,)).fetchall()

print(find_employees(connection, minimum_salary=85000, sort_by="salary"))

[('Asha', 29, 87000.0), ('Krish', 34, 90000.0), ('Meera', 41, 106000.0)]


## 16. Backups and copying a real database for experiments

Never test destructive SQL on the only copy of important data. You can copy the file while it is closed, or use SQLite's `backup()` API while connections are open.

In [18]:
with TemporaryDirectory() as temporary_directory:
    copied_path = Path(temporary_directory) / "sales_copy.db"
    copy2(sales_db_path, copied_path)
    copied_connection = sqlite3.connect(copied_path)
    try:
        copied_count = copied_connection.execute("SELECT COUNT(*) FROM sales").fetchone()[0]
    finally:
        copied_connection.close()
    print("Rows in safe temporary copy:", copied_count)

Rows in safe temporary copy: 5


## 17. Important SQLite concepts and edge cases

- **One writer, many readers:** SQLite is excellent for local apps and moderate workloads, but heavy concurrent writing may need a client/server database.
- **Lock timeout:** `sqlite3.connect(path, timeout=5)` waits briefly for a lock.
- **Foreign keys:** enable them on every connection with `PRAGMA foreign_keys = ON`.
- **Dates:** SQLite has no dedicated date type. Store ISO text (`YYYY-MM-DD`), Unix timestamps, or another consistent representation.
- **Booleans:** commonly stored as `0` and `1`.
- **Transactions:** Python's default behavior starts them for data-changing statements. Understand `isolation_level`; use `None` only when you deliberately want autocommit.
- **Threading:** do not casually share one connection across threads. Prefer one connection per unit of work.
- **Large results:** iterate the cursor or use `fetchmany()` instead of `fetchall()`.
- **Schema changes:** use migrations and backups for real projects.

## 18. Common mistakes

| Mistake | Safer habit |
|---|---|
| f-string values inside SQL | use `?` placeholders |
| forgetting `commit()` | commit intentionally after successful writes |
| forgetting rollback | rollback when a transaction fails |
| `UPDATE`/`DELETE` without `WHERE` | preview the same filter with `SELECT` |
| using a cursor after close | finish work before closing the connection |
| `= NULL` | use `IS NULL` |
| assuming foreign keys are on | run `PRAGMA foreign_keys = ON` |
| fetching millions of rows | iterate or fetch in chunks |
| testing on the real database | use a backup or temporary copy |

In [19]:
# Clean ending: close the practice database only after every query is finished.
connection.close()
print("Practice connection closed safely.")

Practice connection closed safely.


# Quick Revision Cheat Sheet

## Key syntax

```python
import sqlite3

with sqlite3.connect("app.db") as connection:
    connection.execute("PRAGMA foreign_keys = ON")
    connection.execute(
        "INSERT INTO users (name) VALUES (?)",
        (name,),
    )
    rows = connection.execute(
        "SELECT id, name FROM users WHERE name = ?",
        (name,),
    ).fetchall()
```

## SQL in one glance

```sql
CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT NOT NULL);
INSERT INTO users (name) VALUES (?);
SELECT * FROM users WHERE id > ? ORDER BY name LIMIT 10;
UPDATE users SET name = ? WHERE id = ?;
DELETE FROM users WHERE id = ?;
```

## Remember

- `?` placeholders protect values.
- `commit()` saves; `rollback()` undoes the current transaction.
- `WHERE` filters rows; `HAVING` filters groups.
- `INNER JOIN` keeps matches; `LEFT JOIN` keeps every left row.
- `fetchone`, `fetchmany`, `fetchall`, or cursor iteration retrieve results.
- Close connections, enable foreign keys, bound large queries, and back up real data.